# Alphashape with holes -- full pipeline (clean reference)

Three sections:
1. Find a shared alpha across input samples (candidate alphas + visual confirmation on H&E).
2. Find a shared grid size across input samples (sweep + visual confirmation on H&E).
3. With alpha and grid size manually pinned, compute captured tissue area (outer alphashape minus inside cavities) and run two validations (inequality + H&E visual overlay).

Reference notebook -- not intended to be executed end-to-end. The Inputs cell holds example filenames; output paths are left as `""` so the user fills them in before running.

---
## Section 0 -- Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import spatialdata as spd
import alphashape
from shapely.geometry import MultiPolygon, Polygon, box
from shapely.ops import unary_union
from shapely.vectorized import contains as _shapely_vec_contains
from scipy.spatial import cKDTree
from scipy.ndimage import label as ndi_label
from spatialdata.transformations import get_transformation
from spatialdata.transformations.transformations import Translation
from pathlib import Path
import warnings, logging, time

warnings.filterwarnings('ignore', message='.*DS_Store.*')
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

### Inputs (example filenames -- replace before running)

In [ ]:
# ---- Inputs (example filenames) ----
ZARR_PATH = Path("/Users/xinyliu/Downloads/Rose's Lab/CellCharterClusters")
SF_CSV    = Path("/Users/xinyliu/Downloads/Rose's Lab/Results/20260427_scale_factors_summary.csv")
SAMPLES   = ["RTCyPSCA_1_3", "RTCyPSCA_2_4", "NoTx_2_2", "CyPSCA_1_1"]

# ---- Outputs (fill in before running) ----
OUTDIR               = ""
TAG                  = ""
ALPHA_CANDIDATES_CSV = ""
GRID_SWEEP_CSV       = ""
SUMMARY_CSV          = ""
COMPARISON_CSV       = ""
VALIDATION1_PNG      = ""

### Load SpatialData and define helpers (shared across sections)

In [ ]:
print("Loading SpatialData...")
sdata = spd.read_zarr(ZARR_PATH)
adata = sdata["segmentation_counts"]
print(f"Loaded: {adata.n_obs} cells, {adata.n_vars} genes")

sf_df = pd.read_csv(SF_CSV, comment='#').set_index('slide')
SCALE_PARAMS = {
    slide: {"scalef": row["tissue_hires_scalef"], "mpp": row["microns_per_pixel"]}
    for slide, row in sf_df.iterrows()
}
TISSUE_TO_SAMPLE = (
    adata.obs.groupby("tissue", observed=True)["sample"].first().to_dict()
)
missing = [s for s in SAMPLES if s not in TISSUE_TO_SAMPLE]
if missing:
    raise KeyError(f"Tissue(s) not found in adata: {missing}")

for t in SAMPLES:
    s      = TISSUE_TO_SAMPLE[t]
    scalef = SCALE_PARAMS[s]["scalef"]
    mpp    = SCALE_PARAMS[s]["mpp"]
    n      = (adata.obs["tissue"] == t).sum()
    print(f"  {t:20s}  slide={s}  scalef={scalef:.5f}  mpp={mpp:.4f}  n_cells={n}")

In [ ]:
def get_cells_hires(tissue_name):
    """Cell centroids in slide-wide hires-px frame."""
    s      = TISSUE_TO_SAMPLE[tissue_name]
    scalef = SCALE_PARAMS[s]["scalef"]
    mask   = (adata.obs["tissue"] == tissue_name).values
    return adata.obsm["spatial"][mask] * scalef


def get_scale(tissue_name):
    s = TISSUE_TO_SAMPLE[tissue_name]
    return SCALE_PARAMS[s]["scalef"], SCALE_PARAMS[s]["mpp"]


def get_hires_image_with_offset(tissue_name):
    """Return (img_yxc, tx, ty). tx/ty come from the per-tissue Translation
    under coord system 'downscale_to_hires' so the image lands in the same
    slide-wide hires frame as cell centroids."""
    img = sdata.images[f"{tissue_name}_hires_tissue_image"]
    arr = np.transpose(np.asarray(img), (1, 2, 0))   # (c,y,x) -> (y,x,c)
    seq = get_transformation(img, get_all=True).get("downscale_to_hires")
    tx = ty = 0.0
    for tr in getattr(seq, "transformations", []) if seq is not None else []:
        if isinstance(tr, Translation):
            t = np.asarray(tr.translation, dtype=float)
            ty, tx = float(t[-2]), float(t[-1])
            break
    return arr, tx, ty

---
## Section 1 -- Alpha selection (find best shared alpha across samples)

Two automated candidate methods + manual visual confirmation on H&E.

- **Method 1 (conservative)**: per-sample `alphashape.optimizealpha`. Shared candidates are `min(alpha*)` (loosest wrap all samples agree on) and `percentile(alpha*, 25)` as a tighter alternative.
- **Method 2 (density-based)**: per-sample median nearest-neighbor distance via `cKDTree`. Shared candidate is `1 / median(median_NN)`.

The candidates are starting points -- the actual chosen alpha is decided **visually** in the H&E overlay step at the bottom of this section.

### Method 1 -- per-sample `optimizealpha`

Slow (binary search per tissue). Expect minutes for tissues with > 10k cells.

In [ ]:
m1_records = []
for t in SAMPLES:
    pts = get_cells_hires(t)
    t0  = time.time()
    print(f"  optimizealpha  {t:20s}  n={len(pts)}  ...", flush=True)
    a_star = alphashape.optimizealpha(pts)
    print(f"    alpha* = {a_star:.6f}   ({time.time()-t0:.1f}s)")
    m1_records.append({"sample": t, "n_cells": int(len(pts)),
                       "alpha_optimize": float(a_star)})

m1_df = pd.DataFrame(m1_records).set_index("sample")
m1_df

### Method 2 -- density-based 1/median(NN)

In [ ]:
m2_records = []
for t in SAMPLES:
    pts       = get_cells_hires(t)
    nn_dists  = cKDTree(pts).query(pts, k=2)[0][:, 1]   # col 0 is the point itself
    median_nn = float(np.median(nn_dists))
    m2_records.append({
        "sample":             t,
        "median_NN_hires_px": median_nn,
        "inv_median_NN":      1.0 / median_nn,
    })
    print(f"  {t:20s}  median NN = {median_nn:.3f} hires-px   "
          f"1/median_NN = {1.0/median_nn:.6f}")

m2_df = pd.DataFrame(m2_records).set_index("sample")
m2_df

### Combine into shared-alpha candidate table

In [ ]:
summary = m1_df.join(m2_df)
alpha_list     = summary["alpha_optimize"].to_numpy()
median_NN_list = summary["median_NN_hires_px"].to_numpy()

m1_min   = float(np.min(alpha_list))
m1_p25   = float(np.percentile(alpha_list, 25))
m2_alpha = 1.0 / float(np.median(median_NN_list))

candidates = pd.DataFrame([
    {"method": "method1_min",          "alpha": m1_min,   "description": "min over per-sample optimizealpha"},
    {"method": "method1_percentile25", "alpha": m1_p25,   "description": "25th percentile of per-sample optimizealpha"},
    {"method": "method2_inv_med_NN",   "alpha": m2_alpha, "description": "1 / median across samples of per-sample median NN"},
])
candidates.to_csv(ALPHA_CANDIDATES_CSV, index=False)
print(candidates)

### Visualize candidate alphas on H&E (pick by eye)

Loop `ALPHA_TRY` through the candidate values plus any manual values worth testing. Each entry produces one 2x2 H&E + alphashape outline panel (one cell per sample). The outline is drawn in lime for contrast against the natural H&E pink/purple. After visual review, paste the chosen alpha into the parameter cell at the start of Section 3.

In [ ]:
def _draw_polygon_outline(ax, geom, **kwargs):
    """Draw exterior + interior rings of a Polygon or MultiPolygon as lines."""
    polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        if poly.is_empty:
            continue
        x, y = poly.exterior.xy
        ax.plot(x, y, **kwargs)
        for interior in poly.interiors:
            xi, yi = interior.xy
            ax.plot(xi, yi, **kwargs)


def plot_alphashape_overlay_2x2(samples, alpha, save_path):
    """2x2 panel: H&E + alphashape outline at the given alpha, one sample per cell."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle(f"Alphashape on H&E   |   alpha = {alpha:.5f}",
                 fontsize=11, y=0.995)

    for ax, t in zip(axes.flat, samples):
        pts        = get_cells_hires(t)
        he, tx, ty = get_hires_image_with_offset(t)
        boundary   = alphashape.alphashape(pts, alpha)
        n_polys    = len(boundary.geoms) if isinstance(boundary, MultiPolygon) else 1
        # Keep only the largest polygon for the outline (n_polys above still
        # reports the pre-filter count so alpha-induced fragmentation stays visible).
        if isinstance(boundary, MultiPolygon):
            boundary = max(boundary.geoms, key=lambda p: p.area)

        ax.imshow(he, extent=(tx, tx + he.shape[1], ty + he.shape[0], ty))
        ax.scatter(pts[:, 0], pts[:, 1], s=0.2, alpha=0.18, color="steelblue",
                   rasterized=True, linewidths=0)
        _draw_polygon_outline(ax, boundary, color="lime", linewidth=1.6, zorder=3)
        ax.set_aspect("equal")
        ax.set_title(f"{t}   n_cells={len(pts)}   polygons={n_polys}", fontsize=9)
        ax.set_xlabel("hires x (px)", fontsize=8)
        ax.set_ylabel("hires y (px)", fontsize=8)
        ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# Try the three automated candidates plus any manual values worth comparing.
ALPHA_TRY = [m1_min, m1_p25, m2_alpha]    # add custom floats here, e.g. 0.0356

for a in ALPHA_TRY:
    out_png = ""   # e.g. f"{OUTDIR}/{TAG}_alpha_{a:.5f}_overlay.png"
    plot_alphashape_overlay_2x2(SAMPLES, a, out_png)

---
## Section 2 -- Grid size selection (find best shared grid size across samples)

Sweep a list of grid sizes; for each sample produce one figure with all sweep sizes overlaid as viridis-colored contours over a dim cell scatter. Pick the grid size whose contour traces real cavities (bronchi, vessels, large alveolar voids) on H&E most cleanly, without erasing thin tissue. Paste the chosen grid size into the parameter cell at the start of Section 3.

### Grid occupancy core

In [ ]:
def compute_grid_occupancy(points_hires, grid_size_um, mpp, scalef):
    pts = np.asarray(points_hires, dtype=float)
    grid_size_hires_px = grid_size_um * scalef / mpp
    grid_idx           = np.floor(pts / grid_size_hires_px).astype(np.int64)
    occupied_indices   = np.unique(grid_idx, axis=0)
    occupied_count     = len(occupied_indices)
    area_hires_px2     = occupied_count * grid_size_hires_px**2
    area_um2           = area_hires_px2 / scalef**2 * mpp**2
    area_mm2           = area_um2 / 1e6
    return {
        "occupied_count":     occupied_count,
        "grid_size_um":       grid_size_um,
        "grid_size_hires_px": grid_size_hires_px,
        "area_pixel2":        area_hires_px2,
        "area_um2":           area_um2,
        "area_mm2":           area_mm2,
    }


def _occupancy_grid_2d(points_hires, grid_size_hires_px):
    """Return 2D bool mask + matplotlib extent for imshow/contour."""
    x, y = points_hires[:, 0], points_hires[:, 1]
    x_min, x_max = float(x.min()), float(x.max())
    y_min, y_max = float(y.min()), float(y.max())
    nx = int(np.ceil((x_max - x_min) / grid_size_hires_px)) + 1
    ny = int(np.ceil((y_max - y_min) / grid_size_hires_px)) + 1
    grid = np.zeros((ny, nx), dtype=bool)
    ix = np.floor((x - x_min) / grid_size_hires_px).astype(int)
    iy = np.floor((y - y_min) / grid_size_hires_px).astype(int)
    grid[iy, ix] = True
    extent = (x_min, x_min + nx * grid_size_hires_px,
              y_min, y_min + ny * grid_size_hires_px)
    return grid, extent

### Per-sample sensitivity figure (legend outside axes)

In [ ]:
def plot_grid_sensitivity(points_hires, sweep_grid_um, mpp, scalef,
                          save_path, title=""):
    """Per-sample grid sensitivity figure:
       - cell scatter dimmed gray (alpha=0.08) so it sits behind contours,
       - one viridis-colored contour line per grid size,
       - legend (grid size -> area_mm2) outside the axes on the right.
    """
    pts   = np.asarray(points_hires)
    s_dot = max(0.10, min(0.4, 5000 / max(len(pts), 1)))

    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    ax.scatter(pts[:, 0], pts[:, 1], s=s_dot, alpha=0.08, color='gray',
               rasterized=True, linewidths=0)

    colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(sweep_grid_um)))
    for g, color in zip(sweep_grid_um, colors):
        gh           = g * scalef / mpp
        gridS, extS  = _occupancy_grid_2d(pts, gh)
        r            = compute_grid_occupancy(pts, g, mpp, scalef)
        ax.contour(gridS.astype(float), levels=[0.5], extent=extS,
                   colors=[color], linewidths=1.0)
        ax.plot([], [], color=color, linewidth=1.6,
                label=f"{g:>5.1f} um   {r['area_mm2']:6.2f} mm^2")

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("hires x (px)", fontsize=8)
    ax.set_ylabel("hires y (px)", fontsize=8)
    ax.tick_params(labelsize=7)
    leg = ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.02, 0.5),
                    framealpha=0.95, title="grid size  ->  area",
                    title_fontsize=9, borderaxespad=0.0)
    leg.get_title().set_fontweight('bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

### Run the sweep -- one sensitivity figure per sample, one row per (sample, grid_size) in the CSV

In [ ]:
SWEEP_UM = list(range(1, 51, 5))    # example: [1, 6, 11, ..., 46] -- adjust as needed

sweep_rows = []
for t in SAMPLES:
    pts         = get_cells_hires(t)
    scalef, mpp = get_scale(t)
    out_png     = ""   # e.g. f"{OUTDIR}/{TAG}_{t}_sensitivity.png"
    plot_grid_sensitivity(
        pts, SWEEP_UM, mpp, scalef, out_png,
        title=f"{t}   |   grid sensitivity   |   n_cells={len(pts)}",
    )
    for g in SWEEP_UM:
        r = compute_grid_occupancy(pts, g, mpp, scalef)
        sweep_rows.append({
            "sample":         t,
            "grid_size_um":   g,
            "occupied_count": r["occupied_count"],
            "area_mm2":       r["area_mm2"],
        })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(GRID_SWEEP_CSV, index=False)
sweep_df.head()

---
## Section 3 -- Captured tissue area + validations

After visual review of Sections 1 and 2, paste the chosen alpha and grid size below. Captured area is computed as the outer alphashape polygon minus connected-component cavities of empty grid cells inside the outer polygon (above a min-area threshold).

Two validations:
- **Validation 1 (inequality)**: per sample, `area_grid <= area_with_holes <= area_alphashape_only`.
- **Validation 2 (visual)**: per-sample H&E overlay -- outer ring (lime) + holes (red fill + red stroke). The reviewer's question is whether every red blob lines up with a visible bronchus, vessel lumen, or large alveolar void on H&E.

### Manual parameters (paste from Sections 1 and 2)

In [ ]:
ALPHA             = 0.0    # paste chosen value from Section 1 visual review
GRID_SIZE_UM      = 0.0    # paste chosen value from Section 2 visual review
MIN_HOLE_AREA_MM2 = 0.01   # ~ 100 x 100 um cavity at typical lung cell density

### `alphashape_with_holes` -- compute donut polygon and per-sample areas

Algorithm:
1. Compute outer alphashape polygon (Polygon or MultiPolygon).
2. Build a square grid (cell edge = `grid_size_um`) over the bounding box.
3. Classify each cell -- *occupied* (any centroid floor-bins to it) and *empty_inside* (cell center inside outer polygon AND not occupied).
4. Connected components on `empty_inside` using 4-connectivity (diagonal-only chains do not merge -- conservative).
5. Drop components below `min_hole_area_mm2`; convert each kept component to a polygon via `unary_union(box, ...)`.
6. `donut = outer.difference(unary_union(holes))`.
7. Convert all areas hires-px^2 -> um^2 -> mm^2 via `area_px2 / scalef**2 * mpp**2 / 1e6`.

In [ ]:
def alphashape_with_holes(points, alpha, grid_size_um, min_hole_area_mm2,
                          mpp, scalef):
    """Outer alphashape minus inside cavities. See markdown above for the
    full algorithm. `area_mm2_grid_occupancy` is emitted alongside the
    corrected area so Validation 1 has a perfectly matched grid baseline
    (same grid, same cells -- no separate computation step).
    """
    pts = np.asarray(points, dtype=float)
    if len(pts) < 4:
        raise ValueError(f"Need >= 4 points for alphashape, got {len(pts)}")

    outer_polygon = alphashape.alphashape(pts, alpha)
    if outer_polygon.is_empty:
        raise ValueError("alphashape returned empty geometry")
    # Keep only the largest polygon if alphashape returned a MultiPolygon
    if isinstance(outer_polygon, MultiPolygon):
        outer_polygon = max(outer_polygon.geoms, key=lambda p: p.area)

    grid_size_hires_px = grid_size_um * scalef / mpp
    def hires_px2_to_mm2(area_px2):
        return area_px2 / (scalef ** 2) * (mpp ** 2) / 1e6

    minx, miny, maxx, maxy = outer_polygon.bounds
    nx = int(np.ceil((maxx - minx) / grid_size_hires_px)) + 1
    ny = int(np.ceil((maxy - miny) / grid_size_hires_px)) + 1

    # occupied: centroid floor-bin
    ix_pts = np.floor((pts[:, 0] - minx) / grid_size_hires_px).astype(np.int64)
    iy_pts = np.floor((pts[:, 1] - miny) / grid_size_hires_px).astype(np.int64)
    valid  = (ix_pts >= 0) & (ix_pts < nx) & (iy_pts >= 0) & (iy_pts < ny)
    occupied = np.zeros((ny, nx), dtype=bool)
    occupied[iy_pts[valid], ix_pts[valid]] = True

    # inside: cell center inside outer polygon
    cx = minx + (np.arange(nx) + 0.5) * grid_size_hires_px
    cy = miny + (np.arange(ny) + 0.5) * grid_size_hires_px
    XX, YY = np.meshgrid(cx, cy)
    inside = _shapely_vec_contains(outer_polygon, XX.ravel(), YY.ravel()).reshape(ny, nx)

    empty_inside = inside & (~occupied)

    structure_4conn = np.array([[0, 1, 0],
                                 [1, 1, 1],
                                 [0, 1, 0]], dtype=np.int64)
    labeled, n_components_raw = ndi_label(empty_inside, structure=structure_4conn)

    cell_area_hires_px2 = grid_size_hires_px ** 2
    cell_area_mm2       = hires_px2_to_mm2(cell_area_hires_px2)

    holes = []
    for k in range(1, n_components_raw + 1):
        mask_k    = (labeled == k)
        n_cells_k = int(mask_k.sum())
        if n_cells_k * cell_area_mm2 < min_hole_area_mm2:
            continue
        ys_idx, xs_idx = np.where(mask_k)
        cell_polys = [
            box(minx + ii * grid_size_hires_px,
                miny + jj * grid_size_hires_px,
                minx + (ii + 1) * grid_size_hires_px,
                miny + (jj + 1) * grid_size_hires_px)
            for jj, ii in zip(ys_idx, xs_idx)
        ]
        hole_geom = unary_union(cell_polys)
        if hole_geom.geom_type == "MultiPolygon":
            holes.extend(list(hole_geom.geoms))
        else:
            holes.append(hole_geom)

    donut_polygon = (outer_polygon.difference(unary_union(holes))
                      if holes else outer_polygon)

    return {
        "donut_polygon":           donut_polygon,
        "outer_polygon":           outer_polygon,
        "holes":                   holes,
        "area_mm2_corrected":      hires_px2_to_mm2(donut_polygon.area),
        "area_mm2_outer":          hires_px2_to_mm2(outer_polygon.area),
        "area_mm2_holes_total":    sum(hires_px2_to_mm2(h.area) for h in holes),
        "area_mm2_grid_occupancy": int(occupied.sum()) * cell_area_mm2,
        "n_holes":                 len(holes),
        "grid_size_um":            grid_size_um,
        "grid_size_hires_px":      grid_size_hires_px,
        "n_components_raw":        int(n_components_raw),
        "n_components_kept":       len(holes),
    }

### Main loop -- compute donut polygon per sample, save summary CSV

Geometry is also cached in `RESULTS_CACHE` so Validation 2 can re-plot without recomputing alphashape.

In [ ]:
RESULTS_CACHE = {}     # tissue_name -> dict from alphashape_with_holes(...) + pts
rows = []
for t in SAMPLES:
    pts         = get_cells_hires(t)
    scalef, mpp = get_scale(t)
    sample_id   = TISSUE_TO_SAMPLE[t]
    n           = len(pts)
    print(f"\n{'='*60}\nTissue: {t}  (sample_id={sample_id})  n_cells={n}")

    res = alphashape_with_holes(
        pts, alpha=ALPHA, grid_size_um=GRID_SIZE_UM,
        min_hole_area_mm2=MIN_HOLE_AREA_MM2, mpp=mpp, scalef=scalef,
    )

    print(f"  outer area      : {res['area_mm2_outer']:.4f} mm^2")
    print(f"  corrected area  : {res['area_mm2_corrected']:.4f} mm^2")
    print(f"  grid occupancy  : {res['area_mm2_grid_occupancy']:.4f} mm^2")
    print(f"  holes total     : {res['area_mm2_holes_total']:.4f} mm^2  "
          f"(n_holes_kept={res['n_holes']}, n_components_raw={res['n_components_raw']})")

    RESULTS_CACHE[t] = {**res, "pts": pts, "scalef": scalef, "mpp": mpp,
                        "sample_id": sample_id}
    rows.append({
        "sample_id":               sample_id,
        "tissue_name":             t,
        "n_cells":                 n,
        "alpha":                   ALPHA,
        "grid_size_um":            GRID_SIZE_UM,
        "min_hole_area_mm2":       MIN_HOLE_AREA_MM2,
        "area_mm2_outer":          res["area_mm2_outer"],
        "area_mm2_corrected":      res["area_mm2_corrected"],
        "area_mm2_grid_occupancy": res["area_mm2_grid_occupancy"],
        "area_mm2_holes_total":    res["area_mm2_holes_total"],
        "n_holes":                 res["n_holes"],
        "n_components_raw":        res["n_components_raw"],
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
summary_df

### Validation 1 -- inequality check

Per sample, must satisfy `area_grid <= area_with_holes <= area_alphashape_only`.

- grid is most conservative (jagged inward edge + cavity-aware).
- alphashape-only is most permissive (smooth + cavity-blind).
- with_holes inherits alphashape's smooth outer + grid's cavity detection, so it should sit between them.

The grid baseline is the `area_mm2_grid_occupancy` column emitted by `alphashape_with_holes` itself (same grid, same cells). The alphashape-only baseline is recomputed here from the cached `outer_polygon` so the comparison is self-contained.

In [ ]:
alpha_only_rows = []
for t in SAMPLES:
    cache          = RESULTS_CACHE[t]
    scalef, mpp    = cache["scalef"], cache["mpp"]
    boundary       = cache["outer_polygon"]
    area_mm2_alpha = boundary.area / (scalef ** 2) * (mpp ** 2) / 1e6
    alpha_only_rows.append({
        "tissue_name":              t,
        "area_mm2_alphashape_only": area_mm2_alpha,
    })
alpha_only_df = pd.DataFrame(alpha_only_rows).set_index("tissue_name")

new_part = summary_df.set_index("tissue_name")[[
    "sample_id", "area_mm2_grid_occupancy", "area_mm2_corrected",
    "grid_size_um", "n_holes", "area_mm2_holes_total",
]].rename(columns={
    "area_mm2_grid_occupancy": "area_mm2_grid",
    "area_mm2_corrected":      "area_mm2_alphashape_with_holes",
})
comp = new_part.join(alpha_only_df, how="left")

comp["inequality_holds"] = (
    (comp["area_mm2_grid"] <= comp["area_mm2_alphashape_with_holes"] + 1e-9) &
    (comp["area_mm2_alphashape_with_holes"] <= comp["area_mm2_alphashape_only"] + 1e-9)
)
denom = comp["area_mm2_alphashape_only"] - comp["area_mm2_grid"]
comp["with_holes_relpos"] = np.where(
    denom > 1e-9,
    (comp["area_mm2_alphashape_with_holes"] - comp["area_mm2_grid"]) / denom,
    np.nan,
)

comp = comp.reset_index()[[
    "sample_id", "tissue_name",
    "area_mm2_grid", "area_mm2_alphashape_with_holes", "area_mm2_alphashape_only",
    "inequality_holds", "with_holes_relpos",
    "n_holes", "area_mm2_holes_total", "grid_size_um",
]]
comp.to_csv(COMPARISON_CSV, index=False)
print(comp.round(4).to_string(index=False))

if not comp["inequality_holds"].all():
    bad = comp.loc[~comp["inequality_holds"], "tissue_name"].tolist()
    print(f"\nWARNING: inequality violated in {len(bad)} tissue(s): {bad}")
    print("  -> revisit alpha, grid_size_um, or min_hole_area_mm2.")
else:
    print("\nOK: grid <= alphashape_with_holes <= alphashape_only holds for all tissues.")

### Validation 1 -- 3-bar chart per sample (red asterisk on inequality violations)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
n_t   = len(comp)
xpos  = np.arange(n_t)
width = 0.27

bars_grid  = ax.bar(xpos - width, comp["area_mm2_grid"],
                    width, color="#3a6ea5",
                    label=f"grid ({GRID_SIZE_UM:.1f} um)",
                    edgecolor="black", linewidth=0.6)
bars_holes = ax.bar(xpos, comp["area_mm2_alphashape_with_holes"],
                    width, color="#d97a3a", label="alphashape with holes",
                    edgecolor="black", linewidth=0.6)
bars_alpha = ax.bar(xpos + width, comp["area_mm2_alphashape_only"],
                    width, color="#5a9c5a", label="alphashape only",
                    edgecolor="black", linewidth=0.6)

for bars in (bars_grid, bars_holes, bars_alpha):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h, f"{h:.2f}",
                ha="center", va="bottom", fontsize=7)

ymax = max(comp["area_mm2_alphashape_only"]) * 1.15
for i, ok in enumerate(comp["inequality_holds"]):
    if not ok:
        ax.text(i, ymax * 0.98, "*", ha="center", va="top", color="red",
                fontsize=22, fontweight="bold")

ax.set_xticks(xpos)
ax.set_xticklabels([f"{r['tissue_name']}\n({r['sample_id']})"
                    for _, r in comp.iterrows()], fontsize=8)
ax.set_ylabel("area (mm^2)")
ax.set_title(
    f"Validation 1 -- internal consistency  (alpha={ALPHA}, "
    f"grid={GRID_SIZE_UM:.1f} um, min_hole={MIN_HOLE_AREA_MM2} mm^2)\n"
    f"expected: grid <= alphashape_with_holes <= alphashape_only",
    fontsize=10,
)
ax.legend(fontsize=9, loc="upper right", framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, ymax)
plt.tight_layout()
plt.savefig(VALIDATION1_PNG, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

### Validation 2 -- H&E overlay (anatomical visual check)

For each tissue: H&E (positioned via the per-tissue Translation under coord system `downscale_to_hires`) + cell centroids + outer polygon (lime line) + holes (red fill + red stroke). The reviewer's question is whether every red blob lines up with a visible bronchus, vessel lumen, or large alveolar void on H&E. Any red blob over solid pink tissue is a false-positive cavity (usually means `min_hole_area_mm2` is too low for this tissue's cell density).

In [ ]:
def _fill_polygon(ax, geom, **kwargs):
    polys = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        if poly.is_empty:
            continue
        x, y = poly.exterior.xy
        ax.fill(x, y, **kwargs)


def plot_overlay_HE(tissue_name, save_path):
    cache       = RESULTS_CACHE[tissue_name]
    pts         = cache["pts"]
    sample_id   = cache["sample_id"]
    he, tx, ty  = get_hires_image_with_offset(tissue_name)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(he, extent=(tx, tx + he.shape[1], ty + he.shape[0], ty))

    s_dot = max(0.10, min(0.4, 5000 / max(len(pts), 1)))
    ax.scatter(pts[:, 0], pts[:, 1], s=s_dot, alpha=0.30, color="steelblue",
               rasterized=True, linewidths=0)

    _draw_polygon_outline(ax, cache["outer_polygon"],
                          color="lime", linewidth=1.4, zorder=3)

    for hole in cache["holes"]:
        _fill_polygon(ax, hole, color="red", alpha=0.30, zorder=4, edgecolor="none")
        _draw_polygon_outline(ax, hole, color="red", linewidth=1.0, zorder=5)

    xs_all = np.concatenate([[tx, tx + he.shape[1]], pts[:, 0]])
    ys_all = np.concatenate([[ty, ty + he.shape[0]], pts[:, 1]])
    ax.set_xlim(xs_all.min() - 20, xs_all.max() + 20)
    ax.set_ylim(ys_all.max() + 20, ys_all.min() - 20)   # y-down image convention
    ax.set_aspect("equal")
    ax.set_xlabel("hires x (px)", fontsize=8)
    ax.set_ylabel("hires y (px)", fontsize=8)
    ax.tick_params(labelsize=7)

    title = (
        f"{tissue_name}   (sample_id={sample_id})\n"
        f"alpha={ALPHA}   grid={GRID_SIZE_UM:.1f} um   "
        f"min_hole={MIN_HOLE_AREA_MM2} mm^2   "
        f"n_holes={cache['n_holes']}   "
        f"area_corrected={cache['area_mm2_corrected']:.3f} mm^2"
    )
    ax.set_title(title, fontsize=10)

    legend_handles = [
        plt.Line2D([], [], color="lime", linewidth=1.6, label="outer alphashape"),
        Patch(facecolor="red", alpha=0.30, edgecolor="red",
              label=f"hole (>= {MIN_HOLE_AREA_MM2} mm^2)"),
        plt.Line2D([], [], marker="o", linestyle="", color="steelblue",
                   markersize=4, alpha=0.5, label="cell centroid"),
    ]
    ax.legend(handles=legend_handles, fontsize=8, loc="upper right", framealpha=0.9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


for t in SAMPLES:
    out_png = ""   # e.g. f"{OUTDIR}/{TAG}_{t}_overlay_HE.png"
    plot_overlay_HE(t, out_png)

---
## Interpretation notes

- **Validation 1** is automatic -- `inequality_holds` must be `True` for every tissue. A violation usually means the alpha is not concave enough (outer polygon undercuts the cell cloud), the grid size is too coarse to resolve cavities, or `min_hole_area_mm2` is so low that grid cells inside small voids are kept while the new method removes them.
- **Validation 2** is manual -- the question is whether every red blob lines up with a visible bronchus, vessel lumen, or large alveolar void on H&E. Any red blob over solid pink tissue is a false-positive cavity.
- `min_hole_area_mm2` is the single knob most worth sweeping after the per-tissue visuals are reviewed.